<a href="https://colab.research.google.com/github/agg506u/Plasmid-Copy-Number-Calculator/blob/main/%E3%83%97%E3%83%A9%E3%82%B9%E3%83%9F%E3%83%89DNA%E3%82%B3%E3%83%94%E3%83%BC%E6%95%B0%E8%A8%88%E7%AE%97%E6%A9%9F_ver3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# @title
# Circular dsDNA（プラスミド）コピー数計算ツール
# 配列組成を反映・末端なし（circular）を考慮

from IPython.display import display, HTML
import json

# ===== 計算に必要な定数 =====
base_weights = {'A': 331.2218, 'T': 322.2085, 'G': 347.2212, 'C': 307.1971}
WATER = 18.01528
AVOGADRO = 6.02214076e23

# 定数をJavaScriptに安全に渡すための変換
base_weights_json = json.dumps(base_weights)

# ===== 画面（HTML/JS）の構築 =====
# Pythonのf文字列による干渉を防ぐため、通常の文字列として結合します
html_code = """
<div style="font-family: Arial, sans-serif; max-width: 600px; margin: 10px 0; padding: 15px; border: 1px solid #ccc; border-radius: 8px;">
    <h3 style="font-size: 22px; margin-top: 0; color: #333;">🧬 Circular dsDNA コピー数計算ツール</h3>

    <div style="margin-bottom: 15px;">
        <label style="display: block; font-weight: bold; margin-bottom: 5px;">配列 (5'→3'):</label>
        <textarea id="seqInput" placeholder="ここにプラスミド配列を貼り付けてください" style="width: 100%; height: 120px; box-sizing: border-box; padding: 8px; border: 1px solid #ccc; border-radius: 4px; resize: vertical;"></textarea>
    </div>

    <div style="margin-bottom: 15px;">
        <label style="display: block; font-weight: bold; margin-bottom: 5px;">濃度 (ng/µL):</label>
        <input type="number" id="concInput" value="0.0" step="0.1" style="width: 100px; padding: 6px; border: 1px solid #ccc; border-radius: 4px;">
    </div>

    <div style="margin-bottom: 15px;">
        <button id="calcBtn" style="background-color: #2c7be5; color: white; font-weight: bold; border: none; padding: 10px 20px; border-radius: 4px; cursor: pointer; margin-right: 10px;">計算</button>
        <button id="clearBtn" style="background-color: #dc3545; color: white; font-weight: bold; border: none; padding: 10px 20px; border-radius: 4px; cursor: pointer;">クリア</button>
    </div>

    <div id="resultArea" style="width: 100%; min-height: 150px; box-sizing: border-box; padding: 10px; border: 1px solid #ccc; border-radius: 4px; background-color: #fafafa; white-space: pre-wrap; font-family: monospace; font-size: 14px; color: #333;">ここに結果が表示されます</div>
</div>

<script>
(function() {
    const baseWeights = """ + base_weights_json + """;
    const WATER = """ + str(WATER) + """;
    const AVOGADRO = """ + str(AVOGADRO) + """;

    const seqInput = document.getElementById('seqInput');
    const concInput = document.getElementById('concInput');
    const calcBtn = document.getElementById('calcBtn');
    const clearBtn = document.getElementById('clearBtn');
    const resultArea = document.getElementById('resultArea');

    calcBtn.addEventListener('click', () => {
        try {
            // 改行とスペースの除去
            let seq = seqInput.value.replace(/\\n/g, '').replace(/ /g, '').toUpperCase();
            let conc_ng_uL = parseFloat(concInput.value);

            if (seq.length === 0) {
                throw new Error("配列が空です");
            }
            if (isNaN(conc_ng_uL)) {
                throw new Error("濃度を正しく入力してください");
            }

            // 塩基カウント
            let counts = { 'A': 0, 'T': 0, 'G': 0, 'C': 0 };
            let length = 0;
            for (let char of seq) {
                if (counts.hasOwnProperty(char)) {
                    counts[char]++;
                    length++;
                }
            }

            if (length === 0) {
                throw new Error("有効な塩基（A, T, G, C）が見つかりません");
            }

            // 1本鎖分子量
            let mw_ss = 0;
            for (let b in counts) {
                mw_ss += counts[b] * baseWeights[b];
            }

            // Circular DNA：脱水縮合は N 回
            mw_ss -= WATER * length;

            // dsDNA分子量
            let mw_ds = mw_ss * 2;

            // コピー数計算
            let g_per_uL = conc_ng_uL * 1e-9;
            let moles_per_uL = g_per_uL / mw_ds;
            let copies_per_uL = moles_per_uL * AVOGADRO;

            // 結果表示の整形
            resultArea.innerHTML = `🧬 Circular dsDNA（プラスミド）コピー数計算結果\\n\\n` +
                `📏 長さ: ${length} bp\\n` +
                `🧩 塩基組成:\\n` +
                `    A: ${counts['A']}\\n` +
                `    T: ${counts['T']}\\n` +
                `    G: ${counts['G']}\\n` +
                `    C: ${counts['C']}\\n\\n` +
                `⚖️ 分子量（dsDNA）: ${mw_ds.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2})} g/mol\\n` +
                `🧪 濃度: ${conc_ng_uL.toFixed(3)} ng/µL\\n` +
                `🧾 コピー数: ${copies_per_uL.toExponential(3)} copies/µL`;

        } catch (e) {
            resultArea.innerHTML = `⚠️ エラー: ${e.message}`;
        }
    });

    clearBtn.addEventListener('click', () => {
        seqInput.value = '';
        concInput.value = '0.0';
        resultArea.innerHTML = 'ここに結果が表示されます';
    });
})();
</script>
"""

# 画面に表示
display(HTML(html_code))